# The Rank of a Matrix in General

*Course notes for **Math for Machine Learning**, C1 · W2 · L2 — "The Rank of a Matrix in General" (DeepLearning.AI).*

[Rank](05_rank_of_a_matrix.ipynb) counted independent pieces of information in a 2×2 matrix. The idea extends unchanged to any size: the **rank** is the number of independent equations (independent rows). Here we count it for four 3×3 systems, then meet a quicker way to compute it — via **row echelon form**.

In [ ]:
import numpy as np
from fractions import Fraction as F

## 1. Rank of 3×3 systems

Each system has 3 equations (constants = 0). Count how many carry independent information:

$$
\textbf{1}\begin{cases}a+b+c=0\\a+2b+c=0\\a+b+2c=0\end{cases}
\textbf{2}\begin{cases}a+b+c=0\\a+b+2c=0\\a+b+3c=0\end{cases}
\textbf{3}\begin{cases}a+b+c=0\\2a+2b+2c=0\\3a+3b+3c=0\end{cases}
\textbf{4}\begin{cases}0=0\\0=0\\0=0\end{cases}
$$

| System | Independent equations | Rank |
|--------|----------------------|------|
| 1 | 3 — all independent | **3** |
| 2 | 2 — third depends on the others | **2** |
| 3 | 1 — rows 2, 3 are multiples of row 1 | **1** |
| 4 | 0 — no information | **0** |

In [2]:
systems = {
    "System 1": [[1, 1, 1], [1, 2, 1], [1, 1, 2]],
    "System 2": [[1, 1, 1], [1, 1, 2], [1, 1, 3]],
    "System 3": [[1, 1, 1], [2, 2, 2], [3, 3, 3]],
    "System 4": [[0, 0, 0], [0, 0, 0], [0, 0, 0]],
}
for name, A in systems.items():
    r = np.linalg.matrix_rank(A)
    print(f"{name}: rank = {r}  ({r} piece{'s' if r != 1 else ''} of information)")

System 1: rank = 3  (3 pieces of information)
System 2: rank = 2  (2 pieces of information)
System 3: rank = 1  (1 piece of information)
System 4: rank = 0  (0 pieces of information)


## 2. The easier way: row echelon form

Counting independent rows by eye gets hard for big matrices. The shortcut:

> **Rank = the number of non-zero rows (pivots) in the row echelon form.**

Elimination uses only [singularity-preserving row operations](./C1_W2_L1_row_operations_preserve_singularity.ipynb), so it never changes the rank. Each pivot marks one independent row; rows that collapse to all-zeros were the dependent ones.

In [3]:
def row_echelon(M):
    """Reduce to row echelon form with exact arithmetic."""
    M = [[F(x) for x in row] for row in M]
    rows, cols = len(M), len(M[0])
    pivot = 0
    for col in range(cols):
        piv = next((r for r in range(pivot, rows) if M[r][col] != 0), None)
        if piv is None:
            continue
        M[pivot], M[piv] = M[piv], M[pivot]
        M[pivot] = [x / M[pivot][col] for x in M[pivot]]
        for r in range(pivot + 1, rows):
            M[r] = [M[r][k] - M[r][col] * M[pivot][k] for k in range(cols)]
        pivot += 1
    return M

def rank_from_ref(M):
    ref = row_echelon(M)
    return sum(any(x != 0 for x in row) for row in ref)   # count non-zero rows

for name, A in systems.items():
    ref = row_echelon(A)
    print(f"{name}: REF non-zero rows = {rank_from_ref(A)}  (matches numpy rank {np.linalg.matrix_rank(A)})")
    for row in ref:
        print("      ", [str(x) for x in row])

System 1: REF non-zero rows = 3  (matches numpy rank 3)
       ['1', '1', '1']
       ['0', '1', '0']
       ['0', '0', '1']
System 2: REF non-zero rows = 2  (matches numpy rank 2)
       ['1', '1', '1']
       ['0', '0', '1']
       ['0', '0', '0']
System 3: REF non-zero rows = 1  (matches numpy rank 1)
       ['1', '1', '1']
       ['0', '0', '0']
       ['0', '0', '0']
System 4: REF non-zero rows = 0  (matches numpy rank 0)
       ['0', '0', '0']
       ['0', '0', '0']
       ['0', '0', '0']


## 3. Rank and singularity (3×3)

A $3\times 3$ matrix is **non-singular** only at **full rank** ($\text{rank}=3$). Anything less is singular:

- System 1: rank 3 → non-singular,
- Systems 2–4: rank $< 3$ → singular.

In [4]:
for name, A in systems.items():
    r = np.linalg.matrix_rank(A)
    print(f"{name}: rank={r}  ->  {'non-singular' if r == 3 else 'singular'}")

System 1: rank=3  ->  non-singular
System 2: rank=2  ->  singular
System 3: rank=1  ->  singular
System 4: rank=0  ->  singular


## 4. Takeaways

- **Rank** generalizes directly: it is the number of **independent rows** (independent pieces of information), whatever the matrix size.
- Quick computation: reduce to **row echelon form** and **count the non-zero rows** — each is a pivot.
- The solution-space relation still holds: $\text{rank} = n - \dim(\text{solution space})$.
- Full rank ($\text{rank}=n$) ⇔ **non-singular**; lower rank ⇔ **singular**.

*Companion:* [the rank of a matrix](05_rank_of_a_matrix.ipynb) (2×2 intro) · [systems to matrices / REF](./C1_W2_L1_systems_to_matrices.ipynb).